In [2]:
# install fastkaggle if not available
try: import fastkaggle
except ModuleNotFoundError:
    !pip install -Uq fastkaggle

from fastkaggle import *

这是[Road to the Top](https://www.kaggle.com/code/jhoward/first-steps-road-to-the-top-part-1)系列的第3部分，在此系列中，我展示了用于解决[Paddy Doctor](https://www.kaggle.com/competitions/paddy-disease-classification)竞赛的过程，最终获得了四次第一名的提交成绩。上一个笔记本可在此处获取：[第2部分](https://www.kaggle.com/code/jhoward/first-steps-road-to-the-top-part-1)。


## 内存与梯度累积


首先，我们将重复上次使用的步骤来访问数据并确保所有最新库已安装，同时我们还将获取测试集所需的文件：


In [3]:
comp = 'paddy-disease-classification'
path = setup_comp(comp, install='fastai "timm>=0.6.2.dev0"')
from fastai.vision.all import *
set_seed(42)

tst_files = get_image_files(path/'test_images').sorted()

在本次分析中，我们的目标是训练一个由多个较大模型组成的集成模型，并使用更大尺寸的输入图像。训练此类模型时，最大的挑战通常是 GPU 显存。截至本文撰写时，Kaggle GPU 提供 16280MiB 的显存。我习惯先在自己的家用电脑上调试 notebook，再上传到 Kaggle——但仍然需要确保它在 Kaggle 上能正常运行（尤其是代码竞赛，这是强制要求）。我的家用电脑配备的是 24GiB 显卡，所以在本地跑得通并不代表在 Kaggle 上也没问题。

能够快速尝试几种模型和图像尺寸、确认哪些配置可以成功运行，这一点非常重要。为了加快这个过程，我们可以只取一小部分数据来跑几个短轮次——显存占用不变，但速度会快很多。

一个简单的方法是选取一个文件数量较少的类别。以下是我们的选项：


In [4]:
df = pd.read_csv(path/'train.csv')
df.label.value_counts()

normal                      1764
blast                       1738
hispa                       1594
dead_heart                  1442
tungro                      1088
brown_spot                   965
downy_mildew                 620
bacterial_leaf_blight        479
bacterial_leaf_streak        380
bacterial_panicle_blight     337
Name: label, dtype: int64

<output result="pending" reason="incomplete"/>


In [5]:
trn_path = path/'train_images'/'bacterial_panicle_blight'

现在我们来设置一个 `train` 函数，它与上一个 notebook 中训练步骤非常相似，但有几处重要的不同……

第一点是，我使用了一个 `finetune` 参数来决定是调用 `fine_tune()` 方法，还是 `fit_one_cycle()` 方法——后者更快，因为它不会对头部进行初始微调。当在这个函数中执行微调时，我还让它计算并返回测试集上的 TTA 预测结果，因为稍后我们会对多个模型的 TTA 结果进行集成。另外请注意，`ImageDataLoaders` 这一行中不再有 `seed=42`——这意味着每次调用该函数时，训练集和验证集都会不同。这正是集成所需要的，因为这样每个模型使用的数据会略有差异。

更重要的变化是，我添加了一个 `accum` 参数来实现<em>梯度累积</em>。正如你在下面的代码中所看到的，它做了两件事：

1. 将批次大小除以 `accum`
1. 添加 `GradientAccumulation` 回调，并传入 `accum`。


In [6]:
def train(arch, size, item=Resize(480, method='squish'), accum=1, finetune=True, epochs=12):
    dls = ImageDataLoaders.from_folder(trn_path, valid_pct=0.2, item_tfms=item,
        batch_tfms=aug_transforms(size=size, min_scale=0.75), bs=64//accum)
    cbs = GradientAccumulation(64) if accum else []
    learn = vision_learner(dls, arch, metrics=error_rate, cbs=cbs).to_fp16()
    if finetune:
        learn.fine_tune(epochs, 0.01)
        return learn.tta(dl=dls.test_dl(tst_files))
    else:
        learn.unfreeze()
        learn.fit_one_cycle(epochs, 0.01)

<em>梯度累积</em>是一个非常简单的技巧：与其在每个批次之后根据该批次的梯度更新模型权重，不如先<em>累积</em>（相加）几个批次的梯度，再用这些累积的梯度来更新模型权重。在 fastai 中，传递给 `GradientAccumulation` 的参数定义了要累积多少个批次的梯度。由于我们是将 `accum` 个批次的梯度相加，因此需要将批次大小除以相同的数值。最终的训练循环在数学上与使用原始批次大小几乎完全等价，但内存占用却与批次大小缩小 `accum` 倍时相同！

例如，以下是一个不使用梯度累积的单轮训练循环的基本示例：

```python
for x,y in dl:
    calc_loss(coeffs, x, y).backward()
    coeffs.data.sub_(coeffs.grad * lr)
    coeffs.grad.zero_()
```

下面是同样的示例，但加入了梯度累积（假设目标有效批次大小为 64）：

```python
count = 0            # 跟踪自上次权重更新以来看到的项目数量
for x,y in dl:
    count += len(x)  # 根据此小批量大小更新计数
    calc_loss(coeffs, x, y).backward()
    if count>64:     # 计数大于累积目标，因此执行权重更新
        coeffs.data.sub_(coeffs.grad * lr)
        coeffs.grad.zero_()
        count=0      # 重置计数
```

fastai 中的完整实现只有寥寥几行代码——请参阅[源代码](https://github.com/fastai/fastai/blob/master/fastai/callback/training.py#L26)。

为了直观感受梯度累积的效果，来看一个小模型：


In [6]:
train('convnext_small_in22k', 128, epochs=1, accum=1, finetune=False)

让我们创建一个函数来了解它使用了多少内存，以及如何清除内存以供下次运行：


In [7]:
import gc
def report_gpu():
    print(torch.cuda.list_gpu_processes())
    gc.collect()
    torch.cuda.empty_cache()

In [8]:
report_gpu()

所以使用 `accum=1` 时，GPU 使用了约 5GB 内存。让我们试试 `accum=2`：


In [9]:
train('convnext_small_in22k', 128, epochs=1, accum=2, finetune=False)
report_gpu()

可以看到，RAM 用量现在已经降至 4GB。并没有减少一半，因为还有其他开销（对于更大的模型，这部分开销的占比可能会相对更低）。

我们来试试 `4`：


In [10]:
train('convnext_small_in22k', 128, epochs=1, accum=4, finetune=False)
report_gpu()

内存使用量更低了！


## 检查内存使用情况


我们现在来检查后续训练中各种架构和尺寸的内存占用情况，以确保它们都能在 16GB 内存中运行。对于每种情况，我首先尝试 `accum=1`，如果内存占用超过 16GB，则将其翻倍。结果发现，每种情况都需要设置 `accum=2`。

首先是 `convnext_large`：


In [11]:
train('convnext_large_in22k', 224, epochs=1, accum=2, finetune=False)
report_gpu()

In [12]:
train('convnext_large_in22k', (320,240), epochs=1, accum=2, finetune=False)
report_gpu()

这是 `vit_large`。这个非常接近超出我们在 Kaggle 上拥有的 16280MiB 限制！


In [13]:
train('vit_large_patch16_224', 224, epochs=1, accum=2, finetune=False)
report_gpu()

最后是我们的 `swinv2` 和 `swin` 模型：


In [14]:
train('swinv2_large_window12_192_22k', 192, epochs=1, accum=2, finetune=False)
report_gpu()

In [15]:
train('swin_large_patch4_window7_224', 224, epochs=1, accum=2, finetune=False)
report_gpu()

## 运行模型


使用之前的笔记本，我尝试了一堆不同的架构和预处理方法在小模型上，并挑选了一些看起来不错的。我们将使用一个 `dict` 来列出我们将根据该分析为每个感兴趣的架构使用的预处理方法：


In [ ]:
res = 640,480

In [16]:
models = {
    'convnext_large_in22k': {
        (Resize(res), 224),
        (Resize(res), (320,224)),
    }, 'vit_large_patch16_224': {
        (Resize(480, method='squish'), 224),
        (Resize(res), 224),
    }, 'swinv2_large_window12_192_22k': {
        (Resize(480, method='squish'), 192),
        (Resize(res), 192),
    }, 'swin_large_patch4_window7_224': {
        (Resize(480, method='squish'), 224),
        (Resize(res), 224),
    }
}

当然，我们需要切换到使用完整的训练集！


In [17]:
trn_path = path/'train_images'

现在我们准备训练所有这些模型。请注意，每个模型使用的训练集和验证集各不相同，因此结果不能直接进行比较。

我们将把每个模型在测试集上的 TTA 预测结果依次追加到一个名为 `tta_res` 的列表中。


In [18]:
tta_res = []

for arch,details in models.items():
    for item,size in details:
        print('---',arch)
        print(size)
        print(item.name)
        tta_res.append(train(arch, size, item=item, accum=2)) #, epochs=1))
        gc.collect()
        torch.cuda.empty_cache()

## 集成学习


由于这花了相当长的时间来运行，让我们保存结果，以防出现问题！


In [19]:
save_pickle('tta_res.pkl', tta_res)

`Learner.tta` 返回每行的预测值和目标值。我们只需要预测值：


In [20]:
tta_prs = first(zip(*tta_res))

起初我只是使用上述预测，但后来我在较小模型的实验中意识到 `vit` 比其他所有方法都稍好一些，所以我决定在我的集成中给它双倍权重。我通过简单地将其再次添加到列表中来实现这一点（我们也可以通过使用加权平均来实现）：


In [21]:
tta_prs += tta_prs[2:4]

<em>集成</em>（ensemble）简单来说是指一个模型，它本身是将多个其他模型组合在一起的结果。集成最简单的方法是取每个模型预测结果的平均值：


In [22]:
avg_pr = torch.stack(tta_prs).mean(0)
avg_pr.shape

这就是创建集成模型所需的全部内容！最后，我们复制上一个笔记本中用于创建提交文件的步骤：


In [23]:
dls = ImageDataLoaders.from_folder(trn_path, valid_pct=0.2, item_tfms=Resize(480, method='squish'),
    batch_tfms=aug_transforms(size=224, min_scale=0.75))

In [24]:
idxs = avg_pr.argmax(dim=1)
vocab = np.array(dls.vocab)
ss = pd.read_csv(path/'sample_submission.csv')
ss['label'] = vocab[idxs]
ss.to_csv('subm.csv', index=False)

现在我们可以提交了：


In [31]:
if not iskaggle:
    from kaggle import api
    api.competition_submit_cli('subm.csv', 'part 3 v2', comp)

就这样——在撰写本分析时，这个方案轻松登上了排行榜榜首！以下是我提交的四个版本，每一个都比上一个更好，且每一个都曾排名第一：

<img src="https://user-images.githubusercontent.com/346999/174503966-65005151-8f28-4f8b-b3c3-212cf74014f1.png" width="400">

*编辑：实际上，登上排行榜榜首的那个版本在 Kaggle Notebooks 上运行时超时了，所以我不得不从集成中去掉两次运行。不过准确率的差异非常小。*


从下往上，每个结果分别是：

1. `convnext_small` 训练了 12 个 epoch，使用了 TTA
1. `convnext_large` 以相同方式训练
1. 本笔记本中的集成模型，`vit` 模型未过度加权
1. 本笔记本中的集成模型，`vit` 模型过度加权


## 结论


我希望通过本系列传达的核心要点是：只需极少的代码和高度标准化的方法，就能在图像识别上取得出色的效果；而且只要流程严谨，就能实现显著的性能提升。我们的训练函数（包括数据处理和 TTA）仅需六行代码，再加上七行代码用于集成模型并生成提交文件！

如果您觉得这个 notebook 对您有帮助，请记得点击顶部的小上箭头为其点赞——我很乐意知道我的工作对大家有所裨益，这也能帮助更多人发现它。如有任何问题或建议，欢迎在下方留言——我会阅读每一条评论！


In [8]:
# This is what I use to push my notebook from my home PC to Kaggle

if not iskaggle:
    push_notebook('jhoward', 'scaling-up-road-to-the-top-part-3',
                  title='Scaling Up: Road to the Top, Part 3',
                  file='10-scaling-up-road-to-the-top-part-3.ipynb',
                  competition=comp, private=False, gpu=True)

Kernel version 6 successfully pushed.  Please check progress at https://www.kaggle.com/code/jhoward/scaling-up-road-to-the-top-part-3


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文档已使用 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 进行翻译。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言的原文档应被视为权威来源。对于重要信息，建议寻求专业人工翻译。对于因使用本翻译而产生的任何误解或误读，我们概不负责。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
